## 01. Resume Preprocessing 

In [62]:
!pip install wordninja num2words inflect datasets

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [63]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import wordninja
from num2words import num2words
import inflect
from datasets import load_dataset
from collections import Counter
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import spacy

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')
p = inflect.engine()
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\letha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\letha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\letha\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\letha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\letha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [64]:
df = pd.read_csv("C:\code\project-ky8\Data-mining\data_mining\ResumeDataSet.csv")
df.head(20)

,Category,Resume
0,Data Science,Skills * Programming Languages: Python (pandas...
1,Data Science,Education Details \r\nMay 2013 to May 2017 B.E...
2,Data Science,"Areas of Interest Deep Learning, Control Syste..."
3,Data Science,Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...
4,Data Science,"Education Details \r\n MCA YMCAUST, Faridab..."
5,Data Science,"SKILLS C Basics, IOT, Python, MATLAB, Data Sci..."
6,Data Science,Skills â¢ Python â¢ Tableau â¢ Data Visuali...
7,Data Science,Education Details \r\n B.Tech Rayat and Bahr...
8,Data Science,Personal Skills â¢ Ability to quickly grasp t...
9,Data Science,Expertise â Data and Quantitative Analysis â...


In [65]:
df.shape

(962, 2)

In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 962 entries, 0 to 961
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  962 non-null    object
 1   Resume    962 non-null    object
dtypes: object(2)
memory usage: 15.2+ KB


In [67]:
df = df.drop_duplicates(subset=['Resume'], keep='first')

In [68]:
def preprocessing_data(text, idx):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # remove link
    text = re.sub(r'\b(?:https?://)?(?:www\.)?[a-zA-Z0-9-]+\.(?:com|org|net|edu|gov|co|co\.id|info|biz|us|ca|uk|de|jp)\S*\s*', '', text) # remove link
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '', text) # remove email
    text = re.sub(r'\+?\d[\d -]{8,}\d', '', text) # remove phone number
    text = re.sub(r'\b\d{1,4}[/-]\d{1,2}[/-]\d{1,4}\b', '', text) # remove date -> 01/12/2000
    text = re.sub(r'\b[A-Za-z]+\s\d{4}\b', '', text) # remove date -> May 2013
    text = re.sub(r'[^\w\s]', ' ', text) # punctuation
    text = text.replace("exprience", "experience")
    text = text.replace("matelabs", "matlab")
    text = text.replace("maharashtra", " ")
    text = text.replace("monthscompany", " ")
    text = text.replace("-", " ")

    # convert number to word with the following requirement
    pattern = r'\b(\d+)\s+(year|years|months|month|experience|position|skill)\b'

    def replace_match(match):
        number = match.group(1)
        word = p.number_to_words(number)
        return f"{word} {match.group(2)}"

    # Gantikan angka relevan dengan kata
    text = re.sub(pattern, replace_match, text, flags=re.IGNORECASE)

    # Hapus angka yang tidak relevan
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'\b\w\b', '', text) # remove single character
    text = re.sub(r'[^\w\s]', ' ', text) # punctuation

    # remove stopword and lemmatization
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]

    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    text = ' '.join(tokens)

    # remove word that related to location
    doc = nlp(text)
    tokens = [token.text for token in doc if token.ent_type_ != 'GPE']
    text = ' '.join(tokens)

    # # remove verb
    # doc = nlp(text)
    # non_verbs = [token.text for token in doc if token.pos_ not in ["VERB"]]
    # text = ' '.join(non_verbs)

    print(f"Index: {idx}, Preprocessed Text : {' '.join(tokens[:5])}")
    return text


In [69]:
df['clean_data'] = df.apply(lambda row: preprocessing_data(row['Resume'], row.name), axis=1)

Index: 0, Preprocessed Text : skill programming language python panda
Index: 1, Preprocessed Text : education detail uit rgpv data
Index: 2, Preprocessed Text : area interest deep learning control
Index: 3, Preprocessed Text : skill python sap hana tableau
Index: 4, Preprocessed Text : education detail mca ymcaust faridabad
Index: 5, Preprocessed Text : skill basic iot python matlab
Index: 6, Preprocessed Text : skill python tableau data visualization
Index: 7, Preprocessed Text : education detail tech rayat bahra
Index: 8, Preprocessed Text : personal skill ability quickly grasp
Index: 9, Preprocessed Text : expertise data quantitative analysis decision
Index: 40, Preprocessed Text : technical skill typewriting tora spsseducation
Index: 41, Preprocessed Text : skill window xp m office
Index: 42, Preprocessed Text : education detail ba mumbai university
Index: 43, Preprocessed Text : education detail economics chennai tamil
Index: 45, Preprocessed Text : education detail bba lovely pro

In [70]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 166 entries, 0 to 898
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Category    166 non-null    object
 1   Resume      166 non-null    object
 2   clean_data  166 non-null    object
dtypes: object(3)
memory usage: 5.2+ KB


In [71]:
df['text_length'] = df['clean_data'].apply(lambda x: len(x.split()))
print(df['text_length'].describe())

count     166.000000
mean      295.222892
std       257.114510
min        14.000000
25%       108.000000
50%       229.500000
75%       405.000000
max      1422.000000
Name: text_length, dtype: float64


In [72]:
df[df['text_length'] > 405].count()

Category       42
Resume         42
clean_data     42
text_length    42
dtype: int64

In [73]:
all_words = ' '.join(df['clean_data']).split()
word_freq = Counter(all_words)
# print(word_freq.most_common()[-1000:-5000:-1])
# print(word_freq.most_common(100))

In [74]:
# Tentukan ambang batas untuk menghapus kata-kata yang jarang muncul
# Misalnya, kita akan menghapus kata-kata yang muncul kurang dari 2 kali
threshold = 2
rare_words = [word for word, freq in word_freq.items() if freq <= threshold]

# Fungsi untuk menghapus kata-kata yang jarang muncul dari teks
def remove_rare_words(text, rare_words):
    return ' '.join([word for word in text.split() if word not in rare_words])

# Terapkan fungsi ini ke setiap teks dalam data
cleaned_texts = [remove_rare_words(text, rare_words) for text in df['clean_data']]
df = pd.DataFrame(cleaned_texts, columns=['clean_data'])

In [75]:
df['text_length'] = df['clean_data'].apply(lambda x: len(x.split()))
print(df['text_length'].describe())

count     166.000000
mean      265.283133
std       235.009242
min        12.000000
25%        90.000000
50%       194.500000
75%       363.250000
max      1292.000000
Name: text_length, dtype: float64


In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166 entries, 0 to 165
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   clean_data   166 non-null    object
 1   text_length  166 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 2.7+ KB


In [77]:
df.to_csv('C:\code\project-ky8\Data-mining\data_mining\dataset_resume.csv', index=False)

# Hugging Face Resume Job : https://huggingface.co/datasets/cnamuangtoun/resume-job-description-fit

In [78]:
!pip install datasets
from datasets import load_dataset
import pandas as pd

ds = load_dataset("cnamuangtoun/resume-job-description-fit")

df_train = pd.DataFrame(ds['train'])
df_test = pd.DataFrame(ds['test'])

df_train = df_train.drop(columns=['label'])
df_test = df_test.drop(columns=['label'])

df_cv = pd.concat([df_train['resume_text'], df_test['resume_text']], ignore_index=True)
df_jd = pd.concat([df_train['job_description_text'], df_test['job_description_text']], ignore_index=True)


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable


In [79]:
df_cv.info()

<class 'pandas.core.series.Series'>
RangeIndex: 8000 entries, 0 to 7999
Series name: resume_text
Non-Null Count  Dtype 
--------------  ----- 
8000 non-null   object
dtypes: object(1)
memory usage: 62.6+ KB


In [80]:
train_duplicates = df_cv[df_cv.duplicated()].count()
print(train_duplicates)

7357


In [81]:
df_cv = df_cv.drop_duplicates(keep='first')

In [82]:
df_cv.info()

<class 'pandas.core.series.Series'>
Index: 643 entries, 0 to 6277
Series name: resume_text
Non-Null Count  Dtype 
--------------  ----- 
643 non-null    object
dtypes: object(1)
memory usage: 10.0+ KB


In [83]:
def preprocessing_data1(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # remove link
    text = re.sub(r'\b(?:https?://)?(?:www\.)?[a-zA-Z0-9-]+\.(?:com|org|net|edu|gov|co|co\.id|info|biz|us|ca|uk|de|jp)\S*\s*', '', text) # remove link
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '', text) # remove email
    text = re.sub(r'\+?\d[\d -]{8,}\d', '', text) # remove phone number
    text = re.sub(r'\b\d{1,4}[/-]\d{1,2}[/-]\d{1,4}\b', '', text) # remove date -> 01/12/2000
    text = re.sub(r'\b[A-Za-z]+\s\d{4}\b', '', text) # remove date -> May 2013
    text = re.sub(r'[^\w\s]', ' ', text) # punctuation

    # convert number to word with the following requirement
    pattern = r'\b(\d+)\s+(year|years|months|month|experience|position|skill)\b'

    def replace_match(match):
        number = match.group(1)
        word = p.number_to_words(number)
        return f"{word} {match.group(2)}"

    # Gantikan angka relevan dengan kata
    text = re.sub(pattern, replace_match, text, flags=re.IGNORECASE)

    # Hapus angka yang tidak relevan
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'\b\w\b', '', text) # remove single character

    # remove stopword and lemmatization
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]

    preprocessed_text = ' '.join(tokens)
    # print(f"Index: {idx}, Preprocessed Text : {' '.join(tokens[:5])}")
    return preprocessed_text


# Hugging Face Resume Job : https://huggingface.co/datasets/InferencePrince555/Resume-Dataset?row=0

In [84]:
ds1 = load_dataset("InferencePrince555/Resume-Dataset")
df_cv1 = pd.DataFrame(ds1['train'])
df_cv1 = df_cv1.drop(columns=['input'])
df_cv1.head(20)

,instruction,Resume_test
0,Generate a Resume for a Accountant Job,ACCOUNTANT Professional Summary Results orient...
1,Generate a Resume for a Accountant Job,STAFF ACCOUNTANT Summary Flexible Accountant w...
2,Generate a Resume for a Accountant Job,STAFF ACCOUNTANT Summary Highly analytical and...
3,Generate a Resume for a Accountant Job,SENIOR ACCOUNTANT Summary A highly competent m...
4,Generate a Resume for a Accountant Job,SENIOR ACCOUNTANT Summary 11 years experience ...
5,Generate a Resume for a Accountant Job,FINANCIAL ACCOUNTANT Summary CPA Financial Acc...
6,Generate a Resume for a Accountant Job,CORPORATE ACCOUNTANT Summary Over 15 years of ...
7,Generate a Resume for a Accountant Job,ACCOUNTANT Professional Summary Current Accoun...
8,Generate a Resume for a Accountant Job,ACCOUNTANT Summary Innovative and energetic Ac...
9,Generate a Resume for a Accountant Job,ACCOUNTANT Highlights Microsoft Office Interme...


In [85]:
train_duplicates = df_cv1[df_cv1['Resume_test'].duplicated()].count()
print(train_duplicates)

instruction    818
Resume_test    818
dtype: int64


In [86]:
df_cv1 = df_cv1.drop_duplicates(keep='first')
df_cv1 = df_cv1.dropna(subset=['Resume_test'])

In [87]:
def preprocess_job_title(text):
    cleaned_text = re.sub(r'Generate a Resume for a\s+|\s+Job', '', text, flags=re.IGNORECASE)

    cleaned_text = cleaned_text.lower()

    return cleaned_text

In [88]:
import re
df_cv1['category'] = df_cv1['instruction'].apply(preprocess_job_title)

In [89]:
def preprocess_job_title(text):
    cleaned_text = re.sub(r'Generate a Resume for a\s+|\s+Job', '', text, flags=re.IGNORECASE)

    cleaned_text = cleaned_text.lower()

    return cleaned_text

In [90]:
df_cv1 = df_cv1.drop(columns=['instruction'])
df_cv1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31680 entries, 0 to 32480
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Resume_test  31680 non-null  object
 1   category     31680 non-null  object
dtypes: object(2)
memory usage: 742.5+ KB


In [91]:
df_cv1['category'] = df_cv1['category'].replace('web designing', 'UI/UX Designer')
df_cv1['category'] = df_cv1['category'].replace('database', 'database administrator')

In [92]:
df_cv1['category'].value_counts()

category
software developer           5827
systems administrator        4181
project manager              3527
web developer                3466
database administrator       2795
java developer               2431
python developer             2317
network administrator        2260
security analyst             2259
advocate                      128
sales                         121
hr                            120
information technology        120
business development          119
chef                          118
engineering                   118
accountant                    118
finance                       117
fitness                       117
aviation                      116
consultant                    115
healthcare                    115
banking                       115
construction                  112
public relations              111
arts                          109
designer                      107
teacher                       102
apparel                        97
digit

In [93]:
def preprocessing_data1(text, idx):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # remove link
    text = re.sub(r'\b(?:https?://)?(?:www\.)?[a-zA-Z0-9-]+\.(?:com|org|net|edu|gov|co|co\.id|info|biz|us|ca|uk|de|jp)\S*\s*', '', text) # remove link
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '', text) # remove email
    text = re.sub(r'\+?\d[\d -]{8,}\d', '', text) # remove phone number
    text = re.sub(r'\b\d{1,4}[/-]\d{1,2}[/-]\d{1,4}\b', '', text) # remove date -> 01/12/2000
    text = re.sub(r'\b[A-Za-z]+\s\d{4}\b', '', text) # remove date -> May 2013
    text = re.sub(r'[^\w\s]', ' ', text) # punctuation

    # remove text from this pattern
    pattern = r'.*?\bsummary\b'
    text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # convert number to word with the following requirement
    pattern = r'\b(\d+)\s+(year|years|months|month|experience|position|skill)\b'

    def replace_match(match):
        number = match.group(1)
        word = p.number_to_words(number)
        return f"{word} {match.group(2)}"

    # Gantikan angka relevan dengan kata
    text = re.sub(pattern, replace_match, text, flags=re.IGNORECASE)

    # Hapus angka yang tidak relevan
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'\b\w\b', '', text) # remove single character
    text = re.sub(r'[^\w\s]', ' ', text) # punctuation

    # remove stopword and lemmatization
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]

    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    text = ' '.join(tokens)

    # # remove word that related to location, date, and cardinal
    # doc = nlp(text)
    # tokens = [token.text for token in doc if token.ent_type_ != 'GPE' and token.ent_type_ != 'DATE' and token.ent_type_ != 'CARDINAL' ]
    # text = ' '.join(tokens)

    # # remove verb
    # doc = nlp(text)
    # non_verbs = [token.text for token in doc if token.pos_ not in ["VERB"]]
    # text = ' '.join(non_verbs)

    print(f"Index: {idx}, Preprocessed Text : {' '.join(tokens[:5])}")
    return text


In [94]:
import inflect
from nltk.corpus import stopwords

p = inflect.engine()

df_cv1['clean_data'] = df_cv1.apply(lambda row: preprocessing_data1(row['Resume_test'], row.name), axis=1)

Index: 0, Preprocessed Text : result oriented organized bilingual accounting
Index: 1, Preprocessed Text : flexible accountant adapts seamlessly constantly
Index: 2, Preprocessed Text : highly analytical detail oriented professional
Index: 3, Preprocessed Text : analysis prepare auction sale journal
Index: 4, Preprocessed Text : eleven year experience accounting profession
Index: 5, Preprocessed Text : tool monitor usage minimum guarantee
Index: 6, Preprocessed Text : fifteen year increasingly responsible experience
Index: 7, Preprocessed Text : current accountant city alexandria fifteen
Index: 8, Preprocessed Text : innovative energetic accountant proficient extracting
Index: 9, Preprocessed Text : accountant highlight microsoft office intermediate
Index: 10, Preprocessed Text : staff accountant professional profile gain
Index: 11, Preprocessed Text : highly analytical result driven tax
Index: 12, Preprocessed Text : pursue excellence dynamic business world
Index: 13, Preprocessed Tex

In [95]:
print(df_cv1.columns)

Index(['Resume_test', 'category', 'clean_data'], dtype='object')


In [96]:
df_cv1 = df_cv1.drop(columns=['Resume_test'])
df['text_length'] = df['clean_data'].apply(lambda x: len(x.split()))
print(df['text_length'].describe())

count     166.000000
mean      265.283133
std       235.009242
min        12.000000
25%        90.000000
50%       194.500000
75%       363.250000
max      1292.000000
Name: text_length, dtype: float64


In [97]:
df_cv1.head()

,category,clean_data
0,accountant,result oriented organized bilingual accounting...
1,accountant,flexible accountant adapts seamlessly constant...
2,accountant,highly analytical detail oriented professional...
3,accountant,analysis prepare auction sale journal finalize...
4,accountant,eleven year experience accounting profession b...


In [99]:
df_cv1.to_csv("C:\code\project-ky8\Data-mining\data_mining\dataresume30k.csv")